# 01 — Exploratory Data Analysis

**Competition:** ITU/UN "AI for Good" — A Step Ahead of Drought: Forecasting Global Water Storage
**Purpose:** establish, directly from the raw data and with this notebook as the source of truth, every
empirical claim used to drive the project's design decisions (see `docs/COMPETITIVE_ANALYSIS.md` and
`docs/PROJECT_PLAN.md`). Every number below is computed here, not copied from prior analysis — if a
number in the docs and a number in this notebook ever disagree, the docs are wrong and should be
corrected to match this notebook, not the other way around.

**Reproducibility:** deterministic given the raw data in `data/raw/` (loaded and schema-validated via
`tws_forecast.data.loaders`); no random sampling without a fixed seed. Figures are written to
`notebooks/figures/` as PNG files (referenced by relative path below) rather than embedded inline, so
this notebook's diffs stay small and reviewable in git.

**Structure:**
1. Schema & data types
2. Missing-value audit
3. Target definition verification
4. Grid characteristics
5. Temporal coverage
6. Distributional analysis
7. Baseline reference numbers (global mean / climatology / persistence)
8. Non-stationarity and trend
9. Feature correlation: level vs. delta
10. Spatial correlation (nearest-neighbor, same-month)
11. Summary of key findings


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # non-interactive backend: figures are saved to disk, not embedded inline
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial import cKDTree

# This notebook lives at <repo_root>/notebooks/, so the repo root is one level up.
REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from tws_forecast.data.loaders import load_train, load_test, load_sample_submission

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

FIG_DIR = Path.cwd() / "figures"
FIG_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 42

def savefig(fig, name):
    path = FIG_DIR / name
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved figure: figures/{name}")


In [2]:
train = load_train()
test = load_test()
sub = load_sample_submission()

print(f"Train: {train.shape[0]:,} rows x {train.shape[1]} columns")
print(f"Test:  {test.shape[0]:,} rows x {test.shape[1]} columns")
print(f"SampleSubmission: {sub.shape[0]:,} rows x {sub.shape[1]} columns")
print()
print("All three loaded successfully and passed pandera schema validation")
print("(TRAIN_SCHEMA / TEST_SCHEMA / SAMPLE_SUBMISSION_SCHEMA — see src/tws_forecast/data/contracts.py),")
print("including the full-grid check (exactly 15,715 unique (lat, lon) locations in each raw file).")


Train: 2,154,021 rows x 13 columns
Test:  280,961 rows x 13 columns
SampleSubmission: 280,961 rows x 2 columns

All three loaded successfully and passed pandera schema validation
(TRAIN_SCHEMA / TEST_SCHEMA / SAMPLE_SUBMISSION_SCHEMA — see src/tws_forecast/data/contracts.py),
including the full-grid check (exactly 15,715 unique (lat, lon) locations in each raw file).


## 1. Schema & data types

In [3]:
print("=== Train.csv ===")
display(train.dtypes.to_frame("dtype"))
train.head(3)


=== Train.csv ===
                          dtype
sample_id        string[python]
time             datetime64[ns]
lat                     float64
lon                     float64
TWS_t                   float64
SPEI_01_t               float64
SPEI_03_t               float64
SPEI_06_t               float64
SPEI_12_t               float64
SOIL_MOISTURE_t         float64
month_sin               float64
month_cos               float64
target                  float64


In [4]:
print("=== Test.csv ===")
display(test.dtypes.to_frame("dtype"))
test.head(3)


=== Test.csv ===
                          dtype
ID               string[python]
time             datetime64[ns]
lat                     float64
lon                     float64
TWS_t                   float64
SPEI_01_t               float64
SPEI_03_t               float64
SPEI_06_t               float64
SPEI_12_t               float64
SOIL_MOISTURE_t         float64
month_sin               float64
month_cos               float64
TWS_t_masked               bool


In [5]:
print("=== SampleSubmission.csv ===")
display(sub.dtypes.to_frame("dtype"))
sub.head(3)


=== SampleSubmission.csv ===
                 dtype
ID      string[python]
Target         float64


## 2. Missing-value audit

Confirms the masking mechanism documented in `docs/DATA_DICTIONARY.md`: `TWS_t` is masking's target —
never null in Train.csv, and null in Test.csv exactly where `TWS_t_masked` is True. Every other feature
column is fully populated in both files.


In [6]:
print("Train.csv null counts (all columns):")
train_nulls = train.isna().sum()
display(train_nulls.to_frame("n_null"))
assert train_nulls.sum() == 0, "Train.csv has unexpected nulls"
print("\nConfirmed: zero nulls anywhere in Train.csv.")


Train.csv null counts (all columns):
                 n_null
sample_id             0
time                  0
lat                   0
lon                   0
TWS_t                 0
SPEI_01_t             0
SPEI_03_t             0
SPEI_06_t             0
SPEI_12_t             0
SOIL_MOISTURE_t       0
month_sin             0
month_cos             0
target                0

Confirmed: zero nulls anywhere in Train.csv.


In [7]:
print("Test.csv null counts (all columns):")
test_nulls = test.isna().sum()
display(test_nulls.to_frame("n_null"))

n_masked = int(test["TWS_t_masked"].sum())
pct_masked = 100 * n_masked / len(test)
print(f"\nTWS_t_masked = True for {n_masked:,} / {len(test):,} rows ({pct_masked:.1f}%)")

# The core masking invariant, re-verified here directly (also enforced by TEST_SCHEMA at load time).
mismatch = (test["TWS_t"].isna() != test["TWS_t_masked"]).sum()
print(f"Rows where TWS_t.isna() != TWS_t_masked: {mismatch}  (must be 0)")
assert mismatch == 0
assert test_nulls.drop("TWS_t").sum() == 0, "Unexpected nulls outside TWS_t in Test.csv"
print("Confirmed: TWS_t is the only column with nulls in Test.csv, and nullness exactly equals TWS_t_masked.")


Test.csv null counts (all columns):
                 n_null
ID                    0
time                  0
lat                   0
lon                   0
TWS_t            186913
SPEI_01_t             0
SPEI_03_t             0
SPEI_06_t             0
SPEI_12_t             0
SOIL_MOISTURE_t       0
month_sin             0
month_cos             0
TWS_t_masked          0

TWS_t_masked = True for 186,913 / 280,961 rows (66.5%)
Rows where TWS_t.isna() != TWS_t_masked: 0  (must be 0)
Confirmed: TWS_t is the only column with nulls in Test.csv, and nullness exactly equals TWS_t_masked.


## 3. Target definition verification

Verifies `target[t] == TWS_t[t+1]` at the same location, directly from the raw file rather than assumed
from the competition description. Uses an inner self-join on `(lat, lon, time)` shifted by one calendar
month.


In [8]:
check = train[["lat", "lon", "time", "TWS_t", "target"]].copy()
check["next_month"] = check["time"] + pd.DateOffset(months=1)

merged = check.merge(
    check[["lat", "lon", "time", "TWS_t"]].rename(columns={"time": "next_month", "TWS_t": "TWS_t_next"}),
    on=["lat", "lon", "next_month"],
    how="inner",
)
print(f"Matched {len(merged):,} / {len(check):,} rows to a same-location next-calendar-month row.")

diff = (merged["target"] - merged["TWS_t_next"]).abs()
print(f"max |target - TWS_t(t+1)| across matched rows: {diff.max():.10f}")
print(f"rows with |target - TWS_t(t+1)| > 1e-9: {(diff > 1e-9).sum()}")
assert diff.max() < 1e-9, "target != TWS_t at t+1 for some row — leakage/definition assumption is wrong"
print("\nConfirmed: target is exactly next calendar month's TWS_t at the same location, with zero exceptions.")

unmatched = len(check) - len(merged)
print(f"\n{unmatched:,} rows have no next-month match within Train.csv itself — expected for each")
print("location's final training month (Aug 2015), since its target necessarily lives in Test.csv's")
print("September 2015 masked rows, not in Train.csv.")

del check, merged  # both are full-size (~2M row) copies of train; free them once this section is done


Matched 1,977,398 / 2,154,021 rows to a same-location next-calendar-month row.
max |target - TWS_t(t+1)| across matched rows: 0.0000000000
rows with |target - TWS_t(t+1)| > 1e-9: 0

Confirmed: target is exactly next calendar month's TWS_t at the same location, with zero exceptions.

176,623 rows have no next-month match within Train.csv itself — expected for each
location's final training month (Aug 2015), since its target necessarily lives in Test.csv's
September 2015 masked rows, not in Train.csv.


## 4. Grid characteristics

In [9]:
train_locs = train[["lat", "lon"]].drop_duplicates().reset_index(drop=True)
test_locs = test[["lat", "lon"]].drop_duplicates().reset_index(drop=True)

print(f"Unique (lat, lon) locations in Train.csv: {len(train_locs):,}")
print(f"Unique (lat, lon) locations in Test.csv:  {len(test_locs):,}")

train_set = set(map(tuple, train_locs.values))
test_set = set(map(tuple, test_locs.values))
print(f"Identical location sets (Train == Test): {train_set == test_set}")
print(f"Locations only in Train: {len(train_set - test_set)}")
print(f"Locations only in Test:  {len(test_set - train_set)}")

print(f"\nLatitude range:  [{train_locs['lat'].min():.1f}, {train_locs['lat'].max():.1f}]")
print(f"Longitude range: [{train_locs['lon'].min():.1f}, {train_locs['lon'].max():.1f}]")


Unique (lat, lon) locations in Train.csv: 15,715
Unique (lat, lon) locations in Test.csv:  15,715
Identical location sets (Train == Test): True
Locations only in Train: 0
Locations only in Test:  0

Latitude range:  [-55.5, 83.5]
Longitude range: [-179.5, 179.5]


In [10]:
# Hemisphere / latitude-band breakdown — checks the Northern-Hemisphere-heavy grid imbalance
# claimed in docs/PROJECT_PLAN.md ("5,573 locations 30-60N vs 570 at 60-30S").
locs = train_locs.copy()
locs["hemisphere"] = np.where(locs["lat"] >= 0, "Northern", "Southern")
print("Hemisphere split:")
display(locs["hemisphere"].value_counts().to_frame("n_locations"))

bins = [-90, -60, -30, 0, 30, 60, 90]
labels = ["60-90S", "30-60S", "0-30S", "0-30N", "30-60N", "60-90N"]
locs["lat_band"] = pd.cut(locs["lat"], bins=bins, labels=labels, right=True)
print("\nLatitude-band breakdown:")
band_counts = locs["lat_band"].value_counts().reindex(labels).to_frame("n_locations")
display(band_counts)

n_30_60n = int(band_counts.loc["30-60N", "n_locations"])
n_60_30s = int(band_counts.loc["30-60S", "n_locations"])
print(f"\n30-60N: {n_30_60n:,} locations vs 30-60S: {n_60_30s:,} locations "
      f"(ratio {n_30_60n / n_60_30s:.1f}x) — confirms the documented Northern-Hemisphere imbalance.")


Hemisphere split:
            n_locations
hemisphere             
Northern          12633
Southern           3082

Latitude-band breakdown:
          n_locations
lat_band             
60-90S              0
30-60S            570
0-30S            2512
0-30N            3163
30-60N           5573
60-90N           3897

30-60N: 5,573 locations vs 30-60S: 570 locations (ratio 9.8x) — confirms the documented Northern-Hemisphere imbalance.


In [11]:
fig, ax = plt.subplots(figsize=(11, 5.5))
scatter = ax.scatter(locs["lon"], locs["lat"], c=(locs["lat"] >= 0), cmap="coolwarm",
                      s=3, alpha=0.6)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"Grid: {len(locs):,} land-only locations (blue=Southern, red=Northern hemisphere)")
ax.axhline(0, color="black", linewidth=0.5, linestyle="--")
savefig(fig, "01_grid_map.png")


Saved figure: figures/01_grid_map.png


## 5. Temporal coverage

Checks the exact train/test month ranges and looks directly for any gaps in the monthly cadence — this
is where the "22 missing training months" finding (referenced in `docs/DATA_DICTIONARY.md`) is verified
from scratch.


In [12]:
train_months = sorted(train["time"].unique())
test_months = sorted(test["time"].unique())

print(f"Train: {len(train_months)} distinct months, {pd.Timestamp(train_months[0]).date()} to "
      f"{pd.Timestamp(train_months[-1]).date()}")
print(f"Test:  {len(test_months)} distinct months, {pd.Timestamp(test_months[0]).date()} to "
      f"{pd.Timestamp(test_months[-1]).date()}")

expected_train_months = pd.date_range(train_months[0], train_months[-1], freq="MS")
missing_train_months = sorted(set(expected_train_months) - set(train_months))
print(f"\nExpected {len(expected_train_months)} consecutive months in the train span; "
      f"found {len(train_months)}; missing {len(missing_train_months)}.")
if missing_train_months:
    print("Missing training months:")
    for m in missing_train_months:
        print(f"  {m.date()}")


Train: 138 distinct months, 2002-05-01 to 2015-08-01
Test:  18 distinct months, 2015-09-01 to 2018-12-01

Expected 160 consecutive months in the train span; found 138; missing 22.
Missing training months:
  2002-06-01
  2002-07-01
  2002-08-01
  2003-06-01
  2003-07-01
  2011-01-01
  2011-02-01
  2011-06-01
  2011-07-01
  2012-05-01
  2012-06-01
  2012-10-01
  2012-11-01
  2013-03-01
  2013-04-01
  2013-08-01
  2013-09-01
  2013-10-01
  2014-02-01
  2014-03-01
  2014-07-01
  2014-08-01


In [13]:
expected_test_months = pd.date_range(test_months[0], test_months[-1], freq="MS")
missing_test_months = sorted(set(expected_test_months) - set(test_months))
print(f"Test span {test_months[0]} to {test_months[-1]} could hold {len(expected_test_months)} months; "
      f"Test.csv has {len(test_months)}; {len(missing_test_months)} calendar months are absent entirely "
      "(not just masked — genuinely not present as rows).")
print("\nMonths present in Test.csv:")
for m in test_months:
    print(f"  {pd.Timestamp(m).date()}")
print("\nCalendar months in the test span with NO rows at all in Test.csv:")
for m in missing_test_months:
    print(f"  {m.date()}")


Test span 2015-09-01 00:00:00 to 2018-12-01 00:00:00 could hold 40 months; Test.csv has 18; 22 calendar months are absent entirely (not just masked — genuinely not present as rows).

Months present in Test.csv:
  2015-09-01
  2016-01-01
  2016-02-01
  2016-03-01
  2016-06-01
  2016-07-01
  2016-08-01
  2016-09-01
  2016-12-01
  2017-01-01
  2017-02-01
  2017-03-01
  2017-04-01
  2017-05-01
  2017-06-01
  2018-07-01
  2018-11-01
  2018-12-01

Calendar months in the test span with NO rows at all in Test.csv:
  2015-10-01
  2015-11-01
  2015-12-01
  2016-04-01
  2016-05-01
  2016-10-01
  2016-11-01
  2017-07-01
  2017-08-01
  2017-09-01
  2017-10-01
  2017-11-01
  2017-12-01
  2018-01-01
  2018-02-01
  2018-03-01
  2018-04-01
  2018-05-01
  2018-06-01
  2018-08-01
  2018-09-01
  2018-10-01


In [14]:
fig, ax = plt.subplots(figsize=(13, 2.2))
all_months = pd.date_range(train_months[0], test_months[-1], freq="MS")
present_train = set(train_months)
present_test = set(test_months)

for m in all_months:
    if m in present_train:
        color = "#2c7bb6"
    elif m in present_test:
        color = "#fdae61"
    else:
        color = "#d7191c"
    ax.axvline(m, color=color, linewidth=1.2)

ax.set_yticks([])
ax.set_xlim(all_months[0], all_months[-1])
ax.set_title("Monthly coverage: blue=Train present, orange=Test present, red=absent entirely")
savefig(fig, "02_temporal_coverage.png")


Saved figure: figures/02_temporal_coverage.png


## 6. Distributional analysis

In [15]:
numeric_cols = ["TWS_t", "SPEI_01_t", "SPEI_03_t", "SPEI_06_t", "SPEI_12_t", "SOIL_MOISTURE_t", "target"]
print("Train.csv descriptive statistics:")
display(train[numeric_cols].describe().T)


Train.csv descriptive statistics:
                     count      mean       std  ...       50%       75%       max
TWS_t            2154021.0  0.116995  0.913025  ...  0.131532  0.785811  4.057275
SPEI_01_t        2154021.0 -0.030368  0.956504  ... -0.055874  0.666071  3.475902
SPEI_03_t        2154021.0 -0.055324  0.944398  ... -0.076110  0.631382  4.177270
SPEI_06_t        2154021.0 -0.070566  0.939890  ... -0.089408  0.609807  4.850391
SPEI_12_t        2154021.0 -0.093270  0.930651  ... -0.113459  0.574957  3.764673
SOIL_MOISTURE_t  2154021.0  0.000779  0.787456  ...  0.000655  0.502670  5.385165
target           2154021.0  0.112504  0.911962  ...  0.127302  0.780707  4.057275

[7 rows x 8 columns]


In [16]:
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for ax, col in zip(axes.flat, numeric_cols):
    ax.hist(train[col], bins=80, color="#2c7bb6", alpha=0.8)
    ax.set_title(col)
    ax.axvline(train[col].mean(), color="red", linewidth=1, linestyle="--", label="mean")
axes.flat[-1].axis("off")
axes.flat[-2].legend()
fig.suptitle("Train.csv feature distributions", y=1.01)
fig.tight_layout()
savefig(fig, "03_distributions.png")


Saved figure: figures/03_distributions.png


In [17]:
# Per-location structure: do locations differ meaningfully in level and spread, or is TWS_t essentially
# a global standardized anomaly with no real per-location signal?
loc_stats = train.groupby(["lat", "lon"])["TWS_t"].agg(["mean", "std"])
print(f"Std of per-location means: {loc_stats['mean'].std():.3f}")
print(f"Std of per-location stds:  {loc_stats['std'].std():.3f}")
print(f"Range of per-location stds: [{loc_stats['std'].min():.3f}, {loc_stats['std'].max():.3f}]")
print("\nConfirms real cross-location structure in both level and spread — TWS_t is not a uniform")
print("standardized-anomaly series with identical statistics everywhere.")


Std of per-location means: 0.393
Std of per-location stds:  0.166
Range of per-location stds: [0.284, 1.218]

Confirms real cross-location structure in both level and spread — TWS_t is not a uniform
standardized-anomaly series with identical statistics everywhere.


## 7. Baseline reference numbers

Reproduces the reference RMSE table in `docs/COMPETITIVE_ANALYSIS.md` §3, computed here directly against
Train.csv (in-sample; an order-of-magnitude anchor rather than a true out-of-sample number, since
Test.csv's target is withheld by the competition).


In [18]:
def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))

y = train["target"].values

# Baseline 1: global mean predictor
global_mean_rmse = rmse(y, np.full_like(y, y.mean()))

# Baseline 2: per-location, per-calendar-month climatology (in-sample; a later CV-safe version
# belongs in Project Phase 3's baseline suite, not here — this notebook is descriptive EDA).
# Grouped by an external month key rather than train.copy() + a new column, so this doesn't
# duplicate the full ~2M-row frame just to compute one aggregate.
_month_key = train["time"].dt.month
clim_means = train.groupby([train["lat"], train["lon"], _month_key])["target"].transform("mean")
climatology_rmse = rmse(y, clim_means.values)
del _month_key, clim_means

# Baseline 3: naive persistence (predict "no change" from the current observation)
persistence_rmse = rmse(y, train["TWS_t"].values)

results = pd.DataFrame({
    "Approach": ["Global mean predictor", "Per-location per-month climatology", "Naive persistence (target = TWS_t)"],
    "RMSE": [global_mean_rmse, climatology_rmse, persistence_rmse],
})
display(results)

print(f"\nTarget std (== global mean RMSE by construction): {y.std():.3f}")


                             Approach      RMSE
0               Global mean predictor  0.911962
1  Per-location per-month climatology  0.817274
2  Naive persistence (target = TWS_t)  0.572407

Target std (== global mean RMSE by construction): 0.912


## 8. Non-stationarity and trend

In [19]:
train["year"] = train["time"].dt.year
yearly_mean_target = train.groupby("year")["target"].mean()
display(yearly_mean_target.to_frame("mean_target"))

fig, ax = plt.subplots(figsize=(11, 4.5))
yearly_mean_target.plot(ax=ax, marker="o", color="#2c7bb6")
ax.axhline(0, color="black", linewidth=0.5)
ax.set_ylabel("Mean target (TWS anomaly, t+1)")
ax.set_title("Mean target by year — non-stationary trend")
savefig(fig, "04_yearly_trend.png")

print(f"\n{yearly_mean_target.index.min()}: {yearly_mean_target.iloc[0]:+.3f}")
print(f"Trough (2012-2015 window): {yearly_mean_target.loc[2012:2015].min():+.3f}")
print("Confirms a genuine multi-year drift in the target's mean, not stationary noise around a fixed")
print("level — directly relevant to why random K-fold CV would leak future-period information.")


      mean_target
year             
2002     0.226179
2003     0.139207
2004     0.321941
2005     0.272470
2006     0.157349
2007     0.193019
2008     0.159615
2009     0.069562
2010     0.029533
2011     0.121090
2012    -0.108508
2013    -0.088401
2014     0.020904
2015    -0.135943
Saved figure: figures/04_yearly_trend.png

2002: +0.226
Trough (2012-2015 window): -0.136
Confirms a genuine multi-year drift in the target's mean, not stationary noise around a fixed
level — directly relevant to why random K-fold CV would leak future-period information.


In [20]:
# Persistence RMSE by year — the "2015 anomaly" hard gate flagged in docs/PROJECT_PLAN.md.
train["persist_resid"] = train["target"] - train["TWS_t"]
persistence_rmse_by_year = train.groupby("year").apply(
    lambda g: rmse(g["target"], g["TWS_t"]), include_groups=False
)
display(persistence_rmse_by_year.to_frame("persistence_rmse"))

fig, ax = plt.subplots(figsize=(11, 4.5))
persistence_rmse_by_year.plot(ax=ax, marker="o", color="#d7191c")
ax.set_ylabel("Persistence RMSE")
ax.set_title("Naive persistence RMSE by year")
savefig(fig, "05_persistence_rmse_by_year.png")

print(f"\n2015 persistence RMSE: {persistence_rmse_by_year.loc[2015]:.3f}")
print(f"2002-2014 range: [{persistence_rmse_by_year.loc[2002:2014].min():.3f}, "
      f"{persistence_rmse_by_year.loc[2002:2014].max():.3f}]")
print("\nReproduces the documented anomaly: 2015 persistence RMSE is well outside the 2002-2014 band.")
print("This notebook only reproduces the finding; resolving WHY is Project Phase 1, Experiment 2 —")
print("explicitly out of scope here and not yet investigated.")


      persistence_rmse
year                  
2002          0.626768
2003          0.594762
2004          0.584847
2005          0.505733
2006          0.540324
2007          0.524874
2008          0.513005
2009          0.522774
2010          0.536782
2011          0.563960
2012          0.567815
2013          0.515432
2014          0.548108
2015          0.897760
Saved figure: figures/05_persistence_rmse_by_year.png

2015 persistence RMSE: 0.898
2002-2014 range: [0.506, 0.627]

Reproduces the documented anomaly: 2015 persistence RMSE is well outside the 2002-2014 band.
This notebook only reproduces the finding; resolving WHY is Project Phase 1, Experiment 2 —
explicitly out of scope here and not yet investigated.


## 9. Feature correlation: level vs. delta

Checks how much predictive signal each feature has for the target directly (the "level"), versus for
`target - TWS_t` (the "delta", i.e. what's left to explain after naive persistence).


In [21]:
delta = train["target"] - train["TWS_t"]
feature_cols = ["TWS_t", "SPEI_01_t", "SPEI_03_t", "SPEI_06_t", "SPEI_12_t", "SOIL_MOISTURE_t"]

level_corr = train[feature_cols].corrwith(train["target"])
delta_corr = train[feature_cols].corrwith(delta)

corr_table = pd.DataFrame({"corr_with_level_target": level_corr, "corr_with_delta": delta_corr})
corr_table = corr_table.sort_values("corr_with_level_target", ascending=False)
display(corr_table)

print(f"\nTWS_t vs target (level): r = {level_corr['TWS_t']:.3f}")
print(f"TWS_t vs delta:          r = {delta_corr['TWS_t']:.3f}  (mean-reversion signature)")
print(f"Best SPEI feature vs delta: {delta_corr.drop('TWS_t').abs().idxmax()} "
      f"(r = {delta_corr.drop('TWS_t')[delta_corr.drop('TWS_t').abs().idxmax()]:.3f})")


                 corr_with_level_target  corr_with_delta
TWS_t                          0.803260        -0.315315
SPEI_12_t                      0.379573        -0.035157
SPEI_06_t                      0.368248        -0.004886
SOIL_MOISTURE_t                0.320483        -0.058081
SPEI_03_t                      0.285371        -0.064948
SPEI_01_t                      0.203882        -0.050309

TWS_t vs target (level): r = 0.803
TWS_t vs delta:          r = -0.315  (mean-reversion signature)
Best SPEI feature vs delta: SPEI_03_t (r = -0.065)


In [22]:
# SPEI inter-timescale collinearity (SPEI_01 vs SPEI_12, etc.)
spei_cols = ["SPEI_01_t", "SPEI_03_t", "SPEI_06_t", "SPEI_12_t"]
spei_corr = train[spei_cols].corr()
display(spei_corr)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(spei_corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
ax.set_title("SPEI timescale collinearity")
savefig(fig, "06_spei_correlation.png")


           SPEI_01_t  SPEI_03_t  SPEI_06_t  SPEI_12_t
SPEI_01_t   1.000000   0.595711   0.430912   0.295543
SPEI_03_t   0.595711   1.000000   0.704158   0.494342
SPEI_06_t   0.430912   0.704158   1.000000   0.697698
SPEI_12_t   0.295543   0.494342   0.697698   1.000000
Saved figure: figures/06_spei_correlation.png


## 10. Spatial correlation (nearest-neighbor, same-month)

For each location, finds its single nearest neighbor by great-circle-ish planar distance on (lat, lon),
then measures how correlated their `TWS_t` values are within the *same* calendar month, aggregated
across a sample of months for a stable estimate.


In [23]:
# Nearest neighbor per location (grid is fixed across time, so this is computed once on unique locations).
coords = locs[["lat", "lon"]].values
tree = cKDTree(coords)
# k=2: the nearest point to itself (distance 0) plus the true nearest neighbor.
dist, idx = tree.query(coords, k=2)
nn_idx = idx[:, 1]

neighbor_map = pd.DataFrame({
    "lat": locs["lat"].values,
    "lon": locs["lon"].values,
    "nn_lat": locs["lat"].values[nn_idx],
    "nn_lon": locs["lon"].values[nn_idx],
})

rng = np.random.default_rng(RANDOM_SEED)
sample_months = rng.choice(train_months, size=min(24, len(train_months)), replace=False)

pairs = []
for m in sample_months:
    month_slice = train.loc[train["time"] == m, ["lat", "lon", "TWS_t"]]
    merged = neighbor_map.merge(month_slice, on=["lat", "lon"], how="inner")
    merged = merged.merge(
        month_slice.rename(columns={"lat": "nn_lat", "lon": "nn_lon", "TWS_t": "TWS_t_neighbor"}),
        on=["nn_lat", "nn_lon"], how="inner",
    )
    pairs.append(merged[["TWS_t", "TWS_t_neighbor"]])

pairs_df = pd.concat(pairs, ignore_index=True)
nn_corr = pairs_df["TWS_t"].corr(pairs_df["TWS_t_neighbor"])
print(f"Nearest-neighbor same-month TWS_t correlation, pooled across {len(sample_months)} sampled months "
      f"({len(pairs_df):,} location-month pairs): r = {nn_corr:.3f}")


Nearest-neighbor same-month TWS_t correlation, pooled across 24 sampled months (373,678 location-month pairs): r = 0.982


In [24]:
fig, ax = plt.subplots(figsize=(6, 6))
sample_plot = pairs_df.sample(min(20_000, len(pairs_df)), random_state=RANDOM_SEED)
ax.scatter(sample_plot["TWS_t"], sample_plot["TWS_t_neighbor"], s=2, alpha=0.15, color="#2c7bb6")
ax.plot([-6, 6], [-6, 6], color="red", linewidth=1, linestyle="--")
ax.set_xlabel("TWS_t (location)")
ax.set_ylabel("TWS_t (nearest neighbor, same month)")
ax.set_title(f"Nearest-neighbor same-month correlation: r = {nn_corr:.3f}")
savefig(fig, "07_nn_correlation.png")

print("\nHigh same-month spatial correlation confirmed. Per docs/COMPETITIVE_ANALYSIS.md §3, this is")
print("close to unusable in the blackout regime specifically, since during a blackout month the entire")
print("grid loses observation together — a masked cell's neighbor is masked too. That claim is about")
print("test-set blackout structure and is verified separately in notebooks/02_forecastability.ipynb,")
print("not here.")


Saved figure: figures/07_nn_correlation.png

High same-month spatial correlation confirmed. Per docs/COMPETITIVE_ANALYSIS.md §3, this is
close to unusable in the blackout regime specifically, since during a blackout month the entire
grid loses observation together — a masked cell's neighbor is masked too. That claim is about
test-set blackout structure and is verified separately in notebooks/02_forecastability.ipynb,
not here.


## 11. Summary of key findings

In [25]:
print("=" * 78)
print("SUMMARY — all figures computed directly in this notebook")
print("=" * 78)
print(f'''
1. Target definition: target[t] == TWS_t[t+1] at the same location, verified exactly
   (max abs diff {diff.max():.2e}).

2. Grid: {len(train_locs):,} unique locations, identical in Train and Test, land-only,
   Northern-Hemisphere-heavy ({n_30_60n:,} at 30-60N vs {n_60_30s:,} at 30-60S).

3. Temporal coverage: Train {pd.Timestamp(train_months[0]).date()} to {pd.Timestamp(train_months[-1]).date()}
   ({len(missing_train_months)} missing calendar months within that span); Test spans
   {pd.Timestamp(test_months[0]).date()} to {pd.Timestamp(test_months[-1]).date()} but contains only
   {len(test_months)} of the {len(expected_test_months)} calendar months in that span.

4. Masking: {pct_masked:.1f}% of Test.csv rows have TWS_t withheld; zero mismatches against
   TWS_t_masked.

5. Baselines (in-sample, Train.csv): global mean {global_mean_rmse:.3f}, climatology
   {climatology_rmse:.3f}, naive persistence {persistence_rmse:.3f}. Persistence is the baseline
   to beat.

6. Non-stationarity: mean target drifts from {yearly_mean_target.iloc[0]:+.3f} in
   {yearly_mean_target.index.min()} to {yearly_mean_target.loc[2012:2015].min():+.3f} in the
   2012-2015 trough — rules out random K-fold CV.

7. 2015 persistence-RMSE anomaly reproduced: {persistence_rmse_by_year.loc[2015]:.3f} vs a
   2002-2014 band of [{persistence_rmse_by_year.loc[2002:2014].min():.3f},
   {persistence_rmse_by_year.loc[2002:2014].max():.3f}]. Investigating the cause is Project Phase 1,
   Experiment 2 (not this notebook).

8. Level-vs-delta correlation collapse: TWS_t r={level_corr['TWS_t']:.3f} on the level but
   r={delta_corr['TWS_t']:.3f} on the delta (mean-reversion); best SPEI feature on delta only
   r={delta_corr.drop('TWS_t').abs().max():.3f}.

9. Nearest-neighbor same-month spatial correlation: r={nn_corr:.3f} — strong, but per
   docs/COMPETITIVE_ANALYSIS.md, expected to be far less useful during blackout months when the
   whole grid loses observation simultaneously (checked in notebook 02).
''')
print("=" * 78)
print("All 9 findings above match docs/PROJECT_PLAN.md and docs/COMPETITIVE_ANALYSIS.md.")
print("No discrepancies found between this notebook's direct computation and the prior written analysis.")
print("=" * 78)


SUMMARY — all figures computed directly in this notebook

1. Target definition: target[t] == TWS_t[t+1] at the same location, verified exactly
   (max abs diff 0.00e+00).

2. Grid: 15,715 unique locations, identical in Train and Test, land-only,
   Northern-Hemisphere-heavy (5,573 at 30-60N vs 570 at 30-60S).

3. Temporal coverage: Train 2002-05-01 to 2015-08-01
   (22 missing calendar months within that span); Test spans
   2015-09-01 to 2018-12-01 but contains only
   18 of the 40 calendar months in that span.

4. Masking: 66.5% of Test.csv rows have TWS_t withheld; zero mismatches against
   TWS_t_masked.

5. Baselines (in-sample, Train.csv): global mean 0.912, climatology
   0.817, naive persistence 0.572. Persistence is the baseline
   to beat.

6. Non-stationarity: mean target drifts from +0.226 in
   2002 to -0.136 in the
   2012-2015 trough — rules out random K-fold CV.

7. 2015 persistence-RMSE anomaly reproduced: 0.898 vs a
   2002-2014 band of [0.506,
   0.627]. Investigatin